<a href="https://colab.research.google.com/github/Chalhotra/ViT-Token-Economy/blob/main/notebooks/01_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Baselines: ViT-Tiny and DeiT-Tiny (ImageNet-100)

This notebook is intentionally thin: it calls into `src/` modules.

## Setup (Colab)

Run this cell to clone the repository. For private repos, you'll need a GitHub token with repo access.

In [ ]:
# If running in a fresh Colab runtime, uncomment these lines:
# !git clone https://github.com/Chalhotra/ViT-Token-Economy.git
# %cd ViT-Token-Economy

In [2]:
!git checkout test-branch

Branch 'test-branch' set up to track remote branch 'test-branch' from 'origin'.
Switched to a new branch 'test-branch'


In [3]:
!pip -q install -r requirements.txt
!pip -q install -e .

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 1.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for vit-deit-baselines (pyproject.toml) ... done


In [4]:
from src.imagenet_mapping import build_imagenet100_to_1k_map
from src.models import ModelConfig, create_model, shrink_imagenet1k_head_to_imagenet100
from src.data import DataConfig, load_imagenet100_split, build_transform_for_model, apply_timm_preprocess, build_loader
from src.eval import evaluate_accuracy_latency_throughput, compute_gflops
from src.utils import get_device, num_params
import torch

In [5]:
device = get_device()
maps = build_imagenet100_to_1k_map()

In [6]:
def run(model_id: str, batch_size: int = 64):
    model = create_model(ModelConfig(model_id=model_id, pretrained=True))
    model = shrink_imagenet1k_head_to_imagenet100(model, maps.new_to_old_map, num_classes=100)
    model = model.to(device).eval()
    ds = load_imagenet100_split(DataConfig(split='validation'))
    transform = build_transform_for_model(model)
    ds_t = apply_timm_preprocess(ds, transform)
    loader = build_loader(ds_t, DataConfig(batch_size=batch_size, split='validation', shuffle=False))
    metrics = evaluate_accuracy_latency_throughput(model, loader, device)
    sample = ds_t[0]['pixel_values'].unsqueeze(0).to(device)
    gflops = compute_gflops(model, sample)
    return {
        'model': model_id,
        'params_m': num_params(model)/1e6,
        'gflops': gflops,
        **metrics
    }

In [ ]:
run('vit_tiny_patch16_224')

In [ ]:
run('deit_tiny_patch16_224')

In [ ]:
!python scripts/run_baseline.py --model vit_tiny_patch16_224 # vit-tiny baseline run

Resolving data files: 100% 17/17 [00:00<00:00, 92963.71it/s]
Resolving data files: 100% 17/17 [00:00<00:00, 111237.39it/s]
Unsupported operator aten::add encountered 25 time(s)
Unsupported operator aten::scaled_dot_product_attention encountered 12 time(s)
Unsupported operator aten::gelu encountered 12 time(s)
The following submodules of the model were never called during the trace of the graph. They may be unused, or they were accessed by direct calls to .forward() or via other python methods. In the latter case they will have zeros for statistics, though their statistics will still contribute to their parent calling module.
blocks.0.attn.attn_drop, blocks.1.attn.attn_drop, blocks.10.attn.attn_drop, blocks.11.attn.attn_drop, blocks.2.attn.attn_drop, blocks.3.attn.attn_drop, blocks.4.attn.attn_drop, blocks.5.attn.attn_drop, blocks.6.attn.attn_drop, blocks.7.attn.attn_drop, blocks.8.attn.attn_drop, blocks.9.attn.attn_drop
Model: vit_tiny_patch16_224
Params: 5.54 M
FLOPs:  1.08 GFLOPs (1x

In [ ]:
!python scripts/run_baseline.py --model deit_tiny_patch16_224 # deit-tiny baseline run

model.safetensors: 100% 22.9M/22.9M [00:01<00:00, 13.1MB/s]  
Resolving data files: 100% 17/17 [00:00<00:00, 115564.29it/s]
Resolving data files: 100% 17/17 [00:00<00:00, 110035.75it/s]
Map: 100% 5000/5000 [01:23<00:00, 59.61 examples/s] 
Unsupported operator aten::add encountered 25 time(s)
Unsupported operator aten::scaled_dot_product_attention encountered 12 time(s)
Unsupported operator aten::gelu encountered 12 time(s)
The following submodules of the model were never called during the trace of the graph. They may be unused, or they were accessed by direct calls to .forward() or via other python methods. In the latter case they will have zeros for statistics, though their statistics will still contribute to their parent calling module.
blocks.0.attn.attn_drop, blocks.1.attn.attn_drop, blocks.10.attn.attn_drop, blocks.11.attn.attn_drop, blocks.2.attn.attn_drop, blocks.3.attn.attn_drop, blocks.4.attn.attn_drop, blocks.5.attn.attn_drop, blocks.6.attn.attn_drop, blocks.7.attn.attn_drop,

In [ ]:
!python scripts/run_baseline.py --model deit_tiny_patch16_224 --topk --reduction-loc "3,6,9" --keep-rate 1.0 # sanity check with keep_rate = 1, should yield nearly same accuracy

Resolving data files: 100% 17/17 [00:00<00:00, 97408.70it/s]
Resolving data files: 100% 17/17 [00:00<00:00, 105791.05it/s]
Unsupported operator aten::add encountered 25 time(s)
Unsupported operator aten::scaled_dot_product_attention encountered 12 time(s)
Unsupported operator aten::gelu encountered 12 time(s)
The following submodules of the model were never called during the trace of the graph. They may be unused, or they were accessed by direct calls to .forward() or via other python methods. In the latter case they will have zeros for statistics, though their statistics will still contribute to their parent calling module.
blocks.0.attn.attn_drop, blocks.1.attn.attn_drop, blocks.10.attn.attn_drop, blocks.11.attn.attn_drop, blocks.2.attn.attn_drop, blocks.3.attn.attn_drop, blocks.4.attn.attn_drop, blocks.5.attn.attn_drop, blocks.6.attn.attn_drop, blocks.7.attn.attn_drop, blocks.8.attn.attn_drop, blocks.9.attn.attn_drop
Model: deit_tiny_patch16_224
Params: 5.54 M
FLOPs:  1.08 GFLOPs (1

In [9]:
!python scripts/run_baseline.py --model vit_tiny_patch16_224 --topk --reduction-loc "3,6,9" --keep-rate 0.9 0.8 0.7 # proper topk functionality


Resolving data files: 100% 17/17 [00:00<00:00, 85087.31it/s]
Resolving data files: 100% 17/17 [00:00<00:00, 133526.53it/s]
Unsupported operator aten::add encountered 25 time(s)
Unsupported operator aten::scaled_dot_product_attention encountered 9 time(s)
Unsupported operator aten::gelu encountered 12 time(s)
Unsupported operator aten::mul encountered 3 time(s)
Unsupported operator aten::softmax encountered 3 time(s)
Unsupported operator aten::mean encountered 3 time(s)
Unsupported operator aten::topk encountered 3 time(s)
The following submodules of the model were never called during the trace of the graph. They may be unused, or they were accessed by direct calls to .forward() or via other python methods. In the latter case they will have zeros for statistics, though their statistics will still contribute to their parent calling module.
blocks.0.attn.attn_drop, blocks.1.attn.attn_drop, blocks.10.attn.attn_drop, blocks.11.attn.attn_drop, blocks.2.attn.attn_drop, blocks.4.attn.attn_drop